# SmartAutoDJ — interactive demo / scratchpad

Drive the real pipeline (`smartautodj.pipeline.run`) end-to-end, **listen** to each
tier in the browser, look at the plots, and compare the objective metrics.

| Tier | What it does |
|------|--------------|
| 1 | baseline naive linear crossfade |
| 2 | tempo-match + downbeat-aligned transition with fade/EQ curves |
| 3 | tier 2 + an additive procedural "bridge" layer (riser / drum fill) |

Run top-to-bottom. Re-run any cell after editing the **knobs** in the first code cell.

> Kernel: use the `smartdj` conda env (`conda activate smartdj`, then
> `python -m ipykernel install --user --name smartdj` if the kernel isn't listed).

## 0. Setup & knobs

In [ ]:
# Make the repo importable whether the notebook is run from notebooks/ or the repo root,
# and whether or not `pip install -e .` has been run.
import os, sys, json
from pathlib import Path

HERE = Path.cwd()
REPO = HERE if (HERE / "src" / "smartautodj").exists() else HERE.parent
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)  # so relative paths (data/, outputs/) resolve like the CLI
print("repo:", REPO)

# ----- knobs: edit these, then re-run the cells below -----
SONG_A   = "data/demo_a.wav"   # outgoing track
SONG_B   = "data/demo_b.wav"   # incoming track
OUT_DIR  = "outputs"
BARS     = 8                   # overlap length in bars (tier 2/3)
BACKEND  = "auto"              # "auto" | "allin1" | "librosa"
BRIDGE   = "riser"             # tier-3 bridge: "riser" | "drum_fill"
SEED     = 0
assert Path(SONG_A).exists() and Path(SONG_B).exists(), "input clips not found — check SONG_A / SONG_B"

In [ ]:
import numpy as np
from IPython.display import Audio, Image, display

from smartautodj import pipeline
from smartautodj import io as io_mod

print("smartautodj loaded from:", pipeline.__file__)

## 1. Listen to the two input tracks

In [ ]:
y_a, sr = io_mod.load_audio(SONG_A)
y_b, _  = io_mod.load_audio(SONG_B)
print(f"A: {len(y_a)/sr:.1f}s   B: {len(y_b)/sr:.1f}s   sr={sr}")
print("Track A (outgoing):"); display(Audio(y_a, rate=sr))
print("Track B (incoming):"); display(Audio(y_b, rate=sr))

## 2. Run all three tiers

Each `run(...)` writes a WAV, a JSON sidecar, and plots into `outputs/`, and returns a
summary dict (`wav`, `sidecar`, `plots`, `metrics`, `tier`).

In [ ]:
results = {}
for tier in (1, 2, 3):
    results[tier] = pipeline.run(
        song_a=SONG_A, song_b=SONG_B, tier=tier, out_dir=OUT_DIR,
        bars=BARS, bridge_kind=BRIDGE, backend=BACKEND, seed=SEED,
    )
    print(f"tier {tier}: {results[tier]['wav']}")

## 3. Listen to each transition

In [ ]:
labels = {1: "Tier 1 — baseline crossfade",
          2: "Tier 2 — beat-aligned",
          3: f"Tier 3 — beat-aligned + {BRIDGE} bridge"}
for tier in (1, 2, 3):
    y, _ = io_mod.load_audio(results[tier]["wav"])
    print(labels[tier]); display(Audio(y, rate=sr))

## 4. Compare the objective metrics

Lower is better for all three. Watch tier 1 → tier 2: beat-alignment error and the
post-stretch BPM gap should drop sharply.

In [ ]:
rows = []
for tier in (1, 2, 3):
    m = results[tier]["metrics"]
    rows.append({
        "tier": tier,
        "beat_err_ms":    round(m["beat_alignment"]["mean_abs_error_sec"] * 1000, 2),
        "max_beat_err_ms": round(m["beat_alignment"]["max_abs_error_sec"] * 1000, 2),
        "bpm_gap_before": m["tempo_match"]["bpm_gap_before"],
        "bpm_gap_after":  m["tempo_match"]["bpm_gap_after"],
        "max_db_jump":    m["loudness_continuity"]["max_db_jump"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index("tier"))
except ImportError:
    # pandas isn't a core dep; fall back to a plain print
    cols = ["tier", "beat_err_ms", "max_beat_err_ms", "bpm_gap_before", "bpm_gap_after", "max_db_jump"]
    print("  ".join(cols))
    for r in rows:
        print("  ".join(f"{r[c]:>14}" for c in cols))

## 5. Look at the plots

Waveforms (A tail / B head / output), fade & EQ curves, the chosen transition region
with beat/downbeat markers, and the spectrogram of the output.

In [ ]:
TIER_TO_SHOW = 2  # change to 1 or 3
for p in results[TIER_TO_SHOW]["plots"]:
    print(os.path.basename(p)); display(Image(filename=p))

## 6. Inspect the JSON sidecar

Everything the pipeline decided — analysis (BPM, beats, downbeats, sections), the
transition plan (region, anchors, stretch ratio, EQ), and metrics — is written next to
every WAV so it's inspectable and reusable by the evaluation code.

In [ ]:
side = json.load(open(results[2]["sidecar"]))
print("keys:", list(side.keys()))
print("\nbackend:", side["analysis_a"]["backend"],
      "| A bpm:", side["analysis_a"]["bpm"],
      "| B bpm:", side["analysis_b"]["bpm"])
print("\nplan:")
print(json.dumps(side["plan"], indent=2))

## 7. Scratch

Free space to try your own clips: drop files into `data/`, set `SONG_A`/`SONG_B`
above, and re-run from section 2. Or call `pipeline.run(...)` directly here with
different `bars`, `backend`, or `bridge_kind`.

In [ ]:
# e.g. a longer, librosa-only tier-2 transition:
# r = pipeline.run(song_a=SONG_A, song_b=SONG_B, tier=2, bars=16, backend="librosa")
# display(Audio(io_mod.load_audio(r["wav"])[0], rate=sr))